In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import pandas as pd
from beartype import beartype


@dataclass(frozen=True)
class IngestConfig:
    raven_tables_dpath: Path
    wavs_dpath: Path
    metadata_xlsx_fpath: Path
    out_csv_fpath: Path

    raven_sep: str = "\t"
    target_channel: int = 1

    # How to link Raven table <-> WAV <-> metadata.
    # file_id is the join key we create everywhere.
    # Common simplest convention: raven file stem == wav stem == metadata wav stem.
    raven_fname_to_file_id: str = "stem"  # "stem" | "parent_stem"
    metadata_file_id_col: str = "file_id"  # set this to the actual column name in your Excel (or leave as "file_id" if you make one)

    # Optional: if your metadata uses a WAV filename column instead of a file_id, set this and we will derive file_id from it.
    metadata_wav_fname_col: str | None = None  # e.g. "WavFile" or "wav_fname"


@beartype
def get_file_id_from_raven_table_fpath(raven_table_fpath: Path, mode: str) -> str:
    if mode == "stem":
        return raven_table_fpath.stem
    if mode == "parent_stem":
        return f"{raven_table_fpath.parent.name}_{raven_table_fpath.stem}"
    raise ValueError(f"Unknown raven_fname_to_file_id mode: {mode}")


@beartype
def read_raven_table(raven_table_fpath: Path, sep: str) -> pd.DataFrame:
    df = pd.read_csv(raven_table_fpath, sep=sep, dtype=str)
    if len(df) == 0:
        return df

    # Raven exports are often strings; convert numerics carefully later.
    df.columns = [c.strip() for c in df.columns]
    return df


@beartype
def normalize_raven_calls_df(raven_df: pd.DataFrame, file_id: str, target_channel: int) -> pd.DataFrame:
    if len(raven_df) == 0:
        return pd.DataFrame(
            columns=[
                "file_id",
                "selection",
                "view",
                "channel",
                "begin_s",
                "end_s",
                "low_hz",
                "high_hz",
                "duration_s",
                "call_type",
            ]
        )

    col_map = {
        "Selection": "selection",
        "View": "view",
        "Channel": "channel",
        "Begin Time (s)": "begin_s",
        "End Time (s)": "end_s",
        "Low Freq (Hz)": "low_hz",
        "High Freq (Hz)": "high_hz",
        "Delta Time (s)": "duration_s",
        "Type": "call_type",
    }

    missing_cols = [c for c in col_map if c not in set(raven_df.columns)]
    if missing_cols:
        raise KeyError(f"Raven table missing expected columns: {missing_cols}. Found columns: {list(raven_df.columns)}")

    df = raven_df[list(col_map.keys())].rename(columns=col_map).copy()
    df.insert(0, "file_id", file_id)

    # Types
    for c in ["selection", "view", "channel"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    for c in ["begin_s", "end_s", "low_hz", "high_hz", "duration_s"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["call_type"] = df["call_type"].astype(str).str.strip()

    # Filter to channel 1 as requested
    n_before = len(df)
    df = df[df["channel"] == target_channel].copy()
    n_after = len(df)
    if n_after == 0 and n_before > 0:
        raise ValueError(
            f"After filtering to channel={target_channel}, no rows remain for file_id={file_id}. "
            f"Check Raven export Channel values and target_channel."
        )

    # Basic sanity checks
    bad_time = df["begin_s"].isna() | df["end_s"].isna() | (df["end_s"] <= df["begin_s"])
    if bad_time.any():
        bad_i = df.index[bad_time].tolist()[:10]
        raise ValueError(f"Found invalid begin/end times for file_id={file_id}. Example row indices: {bad_i}")

    df["duration_s"] = df["end_s"] - df["begin_s"]

    # Stable per-file call index (useful later even if Selection numbers are non-contiguous)
    df = df.sort_values(["begin_s", "end_s", "selection"], kind="mergesort").reset_index(drop=True)
    df.insert(1, "call_i", range(len(df)))

    return df


@beartype
def read_metadata_df(cfg: IngestConfig) -> pd.DataFrame:
    df = pd.read_excel(cfg.metadata_xlsx_fpath)

    df.columns = [str(c).strip() for c in df.columns]

    if cfg.metadata_file_id_col in set(df.columns):
        meta = df.copy()
        meta[cfg.metadata_file_id_col] = meta[cfg.metadata_file_id_col].astype(str).str.strip()
        return meta.rename(columns={cfg.metadata_file_id_col: "file_id"})

    if cfg.metadata_wav_fname_col is None:
        raise KeyError(
            f"Metadata file does not contain '{cfg.metadata_file_id_col}', and metadata_wav_fname_col is None. "
            f"Set cfg.metadata_file_id_col to a real column name or set cfg.metadata_wav_fname_col."
        )

    if cfg.metadata_wav_fname_col not in set(df.columns):
        raise KeyError(
            f"Metadata file missing '{cfg.metadata_wav_fname_col}'. Found columns: {list(df.columns)}"
        )

    meta = df.copy()
    meta[cfg.metadata_wav_fname_col] = meta[cfg.metadata_wav_fname_col].astype(str).str.strip()
    meta["file_id"] = meta[cfg.metadata_wav_fname_col].map(lambda s: Path(s).stem)
    return meta


@beartype
def find_raven_table_fpaths(raven_tables_dpath: Path) -> list[Path]:
    if not raven_tables_dpath.exists():
        raise FileNotFoundError(f"raven_tables_dpath does not exist: {raven_tables_dpath}")

    fpaths = sorted(
        [p for p in raven_tables_dpath.rglob("*") if p.is_file() and p.suffix.lower() in {".txt", ".tsv"}]
    )
    if len(fpaths) == 0:
        raise FileNotFoundError(f"No .txt/.tsv Raven tables found under: {raven_tables_dpath}")

    return fpaths


@beartype
def make_calls_df_from_raven_tables(cfg: IngestConfig) -> pd.DataFrame:
    raven_table_fpaths = find_raven_table_fpaths(cfg.raven_tables_dpath)

    calls_dfs: list[pd.DataFrame] = []
    for raven_table_fpath in raven_table_fpaths:
        file_id = get_file_id_from_raven_table_fpath(raven_table_fpath, cfg.raven_fname_to_file_id)
        raven_df = read_raven_table(raven_table_fpath, cfg.raven_sep)
        calls_df = normalize_raven_calls_df(raven_df, file_id=file_id, target_channel=cfg.target_channel)
        calls_df["raven_table_fpath"] = str(raven_table_fpath)
        calls_dfs.append(calls_df)

    calls = pd.concat(calls_dfs, ignore_index=True)
    return calls


@beartype
def add_wav_paths(calls_df: pd.DataFrame, wavs_dpath: Path) -> pd.DataFrame:
    if not wavs_dpath.exists():
        raise FileNotFoundError(f"wavs_dpath does not exist: {wavs_dpath}")

    wav_map: dict[str, str] = {}
    for wav_fpath in wavs_dpath.rglob("*.wav"):
        wav_map[wav_fpath.stem] = str(wav_fpath)

    if len(wav_map) == 0:
        raise FileNotFoundError(f"No .wav files found under: {wavs_dpath}")

    df = calls_df.copy()
    df["wav_fpath"] = df["file_id"].map(wav_map)

    missing = df["wav_fpath"].isna()
    if missing.any():
        missing_ids = sorted(df.loc[missing, "file_id"].unique().tolist())[:20]
        raise FileNotFoundError(
            f"Some file_id values from Raven tables did not match any WAV stems under {wavs_dpath}. "
            f"Examples: {missing_ids}"
        )

    return df


@beartype
def merge_calls_with_metadata(calls_df: pd.DataFrame, meta_df: pd.DataFrame) -> pd.DataFrame:
    if "file_id" not in set(meta_df.columns):
        raise KeyError("meta_df must include a 'file_id' column after normalization.")

    # Validate uniqueness in metadata join key
    dup = meta_df["file_id"].duplicated(keep=False)
    if dup.any():
        dup_ids = sorted(meta_df.loc[dup, "file_id"].unique().tolist())[:20]
        raise ValueError(f"Metadata has duplicate file_id values. Examples: {dup_ids}")

    merged = calls_df.merge(meta_df, on="file_id", how="left", validate="many_to_one")

    missing = merged.isna().all(axis=1)
    if missing.any():
        raise ValueError("Unexpected fully-missing rows after merge; this suggests a merge/index bug.")

    # Check whether any calls failed to match metadata
    meta_cols = [c for c in meta_df.columns if c != "file_id"]
    if meta_cols:
        no_meta = merged[meta_cols].isna().all(axis=1)
        if no_meta.any():
            bad_ids = sorted(merged.loc[no_meta, "file_id"].unique().tolist())[:20]
            raise ValueError(f"Some file_id values have no matching metadata. Examples: {bad_ids}")

    return merged


@beartype
def setup_make_call_level_csv(cfg: IngestConfig) -> None:
    calls = make_calls_df_from_raven_tables(cfg)
    calls = add_wav_paths(calls, cfg.wavs_dpath)
    meta = read_metadata_df(cfg)
    merged = merge_calls_with_metadata(calls, meta)

    # Add column for the .wav filename
    merged['wav_fname'] = merged['wav_fpath'].map(lambda x: Path(x).name)

    cfg.out_csv_fpath.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(cfg.out_csv_fpath, index=False)


# If you want a simple primitive path helper for notebooks:
@beartype
def get_default_out_csv_fpath(out_dpath: Path, name: str) -> Path:
    if not name.endswith(".csv"):
        name = f"{name}.csv"
    return out_dpath / name


ModuleNotFoundError: No module named 'beartype'

## test if pytorch is running on gpu

In [2]:
import torch

# Check if a GPU is available and set the device accordingly
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available.")
    print(f"Device name: {torch.cuda.get_device_name(0)}") # Prints the name of the first GPU
else:
    device = torch.device("cpu")
    print("No GPU available, using CPU.")

tensor = torch.randn((3, 3)).to(device)

GPU is available.
Device name: GRID A100X-10C


Make call numbers into letters 

In [5]:
# Create a mapping for numbers to letters
import os
import numpy as np
import pandas as pd
df = pd.read_csv('Datasets/cv4e_calls_channel1.csv')
number_to_letter = {
    '1': 'A', '2': 'B', '3': 'C', '4': 'D', '5': 'E',
    '6': 'F', '7': 'G', '8': 'H', '9': 'I', '10': 'J',
    '11': 'K', '12': 'L', '13': 'M', '14': 'N', '15': 'O'
}

# Function to convert call_type: if it's a number (in our mapping), convert to letter; otherwise keep as is
def convert_call_type(call_type):
    return number_to_letter.get(str(call_type).strip(), call_type)

# Apply the conversion
df['call_type'] = df['call_type'].apply(convert_call_type)

# Save the updated dataframe to the new CSV file
df.to_csv('cv4e_calls_channel1_v2.csv', index=False)

print("CSV file saved as cv4e_calls_channel1_v2.csv!")
print("\nUpdated call_type distribution:")
print(df['call_type'].value_counts().sort_values(ascending=True))

CSV file saved as cv4e_calls_channel1_v2.csv!

Updated call_type distribution:
call_type
Growl       5
H          14
A          20
N          20
O          24
L          26
J          39
M          56
I          75
G          77
K         130
B         136
Chits     171
D         201
C         202
E         248
F         256
Cheer     707
Check    3828
Name: count, dtype: int64
